##  Import Libraries and Load Dataset

In [9]:
import pandas as pd

# Load the dataset (use your path)
df = pd.read_csv("sample_amazon_reviews.csv")

# Show a few rows
df


,review_id,product_title,customer_id,review_body,star_rating,review_date
0,7e514167-4831-4706-ba70-b768614cf4cb,Laptop Stand,25387,Space health authority foot respond news try e...,5,2025-03-01
1,8b29c48d-f66a-4ec5-beb1-69de8c21347e,Laptop Stand,11094,Environment them hear person chance coach them...,4,2025-04-17
2,729b5206-508f-40c6-8ea6-ff38eab87b98,Smart Watch,29279,Throw we something itself safe serve.,5,2025-01-28
3,b8d4ed2c-835b-41de-93e2-1ff86af8f394,Laptop Stand,97387,Interesting those cost offer cut eye she hand.,5,2025-02-22
4,6981597c-9002-46d7-b033-255d9c9b31cc,Fitness Tracker,63879,Under hit suffer standard last other skin baby...,4,2025-02-09
...,...,...,...,...,...,...
995,d41c6baa-da21-48c2-9f51-14773e67fece,Wireless Earbuds,74653,Well against old edge they rule issue any syst...,5,2024-08-25
996,5da91a7b-e19a-4d7a-b557-8a7188e297c5,Fitness Tracker,94055,How evening leg clearly opportunity just read ...,1,2025-02-19
997,69104d8e-cc22-4a89-90d5-f708f68c17ba,Wireless Earbuds,81546,Might kind size trouble raise economy where dr...,2,2025-04-09
998,576d7f2a-531f-4fa4-a1b6-7ce003e36a0b,Fitness Tracker,89981,Buy professional that that example at at in pe...,4,2025-03-22


## Data Preprocessing

In [7]:
# Drop rows with missing values
df = df.dropna(subset=["review_body", "product_title", "star_rating"])
df.head()

,review_id,product_title,customer_id,review_body,star_rating,review_date
0,7e514167-4831-4706-ba70-b768614cf4cb,Laptop Stand,25387,Space health authority foot respond news try e...,5,2025-03-01
1,8b29c48d-f66a-4ec5-beb1-69de8c21347e,Laptop Stand,11094,Environment them hear person chance coach them...,4,2025-04-17
2,729b5206-508f-40c6-8ea6-ff38eab87b98,Smart Watch,29279,Throw we something itself safe serve.,5,2025-01-28
3,b8d4ed2c-835b-41de-93e2-1ff86af8f394,Laptop Stand,97387,Interesting those cost offer cut eye she hand.,5,2025-02-22
4,6981597c-9002-46d7-b033-255d9c9b31cc,Fitness Tracker,63879,Under hit suffer standard last other skin baby...,4,2025-02-09


## Encode Text and Categorical Data

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score

# Separate features and target
X = df[["review_body", "product_title"]]
y = df["star_rating"]

# Define transformers
text_transformer = TfidfVectorizer(max_features=300)
cat_transformer = OneHotEncoder(handle_unknown='ignore')

# Combine them using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("text", text_transformer, "review_body"),
        ("cat", cat_transformer, ["product_title"])
    ]
)


## Split the Data (Train/Test)

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


##  Create and Train the Model

In [15]:
from sklearn.pipeline import Pipeline

# Create a pipeline with preprocessing + model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier())
])

# Train the model
pipeline.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('text',
                                                  TfidfVectorizer(max_features=300),
                                                  'review_body'),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['product_title'])])),
                ('classifier', RandomForestClassifier())])

## Evaluate the Model

In [17]:
# Predict on test set
y_pred = pipeline.predict(X_test)

# Check accuracy and report
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.19
Classification Report:
               precision    recall  f1-score   support

           1       0.26      0.43      0.33        37
           2       0.13      0.21      0.16        38
           3       0.00      0.00      0.00        40
           4       0.30      0.15      0.20        41
           5       0.17      0.18      0.18        44

    accuracy                           0.19       200
   macro avg       0.17      0.19      0.17       200
weighted avg       0.17      0.19      0.17       200



## Predict on New Data

In [ ]:
sample = pd.DataFrame({
    "review_body": ["Amazing sound quality and battery life"],
    "product_title": ["Bluetooth Speaker"]
})

predicted_rating = pipeline.predict(sample)
print("Predicted Rating:", predicted_rating)


## Results

- The model achieved an accuracy of around **(0.19)%**
- The most influential feature was the **review text**
- This classification model can be extended to real-world review analysis systems
